# TEDS Score Calculation

json dict -> teds cal

### Creation of Node Class

In [ ]:
import distance
from apted import APTED, Config
 
 
class Node:
    def __init__(self, tag, text=None):
        self.tag = tag
        self.text = text
        self.children = []
 
    def bracket(self):
        inner = "".join(c.bracket() for c in self.children)
        return "{%s|%s%s}" % (self.tag, self.text or "", inner)
 
 
class TableConfig(Config):
    def rename(self, a, b):
        if a.tag != b.tag:
            return 1.0
        if a.tag != "td" or a.text == b.text:
            return 0.0
        if not a.text or not b.text:
            return 1.0
        return distance.levenshtein(a.text, b.text) / max(len(a.text), len(b.text))
 
    def children(self, node):
        return node.children

### Recursive Creation of Tree

In [ ]:
def to_tree(table):
    """Build the tree: table > tr > td."""
    root = Node("table")
    rows = (table.get("headers") and [table["headers"]] or []) + (table.get("rows") or [])
    for row in rows:
        tr = Node("tr")
        tr.children = [Node("td", str(c or "").strip()) for c in row]
        root.children.append(tr)
    return root

## TEDS Score Calculation

In [ ]:
def size(node):
    return 1 + sum(size(c) for c in node.children)
 
 
def teds(pred, ref):
    """TEDS in [0, 1]. A missing prediction scores 0."""
    if not pred:
        return 0.0
    a, b = to_tree(pred), to_tree(ref)
    n = max(size(a), size(b))
    return 1.0 - APTED(a, b, TableConfig()).compute_edit_distance() / n
 